In [ ]:
import re, math, numpy as np, pandas as pd, tensorflow as tf, seaborn as sns, sklearn
from matplotlib import pyplot as plt
from numpy import array
from pandas import read_csv
from scipy.stats import spearmanr, norm, t, chi2

from tensorflow import keras
from tensorflow.keras import backend as K, initializers, regularizers
from tensorflow.keras.models import load_model, Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, LSTM, GRU, SimpleRNN, Conv1D, MaxPooling1D
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
tscv = TimeSeriesSplit()

def split_sequences(sequences, n_steps):
    X, y = list(), list()
    for i in range(len(sequences)):
        end_ix = i + n_steps
        if end_ix > len(sequences) - 1:
            break
        seq_x, seq_y = sequences[i:end_ix, :-1], sequences[end_ix, -1]  
        X.append(seq_x)
        y.append(seq_y)
    return np.array(X), np.array(y)

n_steps = 6
rolling = 504

In [ ]:
GARCH_CSV = "../results/BTC_GARCH.csv"

def load_garch_only(garch_path=GARCH_CSV):
    
    df = pd.read_csv(garch_path)
    
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"])
        df = df.sort_values("Date")
    
    return df

garch_forecasts = load_garch_only()
garch_forecasts.head(3)

In [ ]:
list(garch_forecasts.columns)

In [ ]:
realized = garch_forecasts['SvOLHC'].iloc[-1194:]

models = [
    ("G(N)", "sGARCH_norm_sigma"),
    ("G(STD)", "sGARCH_std_sigma"),
    ("G(SSTD)", "sGARCH_sstd_sigma"),
    ("E(N)", "eGARCH_norm_sigma"),
    ("E(STD)", "eGARCH_std_sigma"),
    ("E(SSTD)", "eGARCH_sstd_sigma"),
    ("GJR(N)", "gjrGARCH_norm_sigma"),
    ("GJR(STD)", "gjrGARCH_std_sigma"),
    ("GJR(SSTD)", "gjrGARCH_sstd_sigma"),
    ("AP(N)", "apARCH_norm_sigma"),
    ("AP(STD)", "apARCH_std_sigma"),
    ("AP(SSTD)", "apARCH_sstd_sigma")
]

results = []

for model_name, col in models:
    if col in garch_forecasts.columns:
        forecast = garch_forecasts[col].iloc[-1194:]
        mse_val = mean_squared_error(realized, forecast)
        results.append((model_name, float(f"{mse_val:.5f}")))

mse_df = pd.DataFrame(results, columns=["Metrics / Model", "MSE"])

print(mse_df)

In [ ]:
def var_normal(mu, sigma, alpha):
    q = norm.ppf(alpha)
    return -(mu + sigma * q)

def es_normal(mu, sigma, alpha):
    q = norm.ppf(alpha)
    return -(mu - sigma * norm.pdf(q)/alpha)

def var_student(mu, sigma, dfree, alpha):
    q = t.ppf(alpha, df=dfree)
    return -(mu + sigma * q)

def es_student(mu, sigma, dfree, alpha):
    q = t.ppf(alpha, df=dfree)
    pdf_q = t.pdf(q, df=dfree)
    c = (dfree + q**2) / (dfree - 1)
    return -(mu - sigma * (pdf_q/alpha) * c)

def compute_var_es(mu, sigma, shape, base, alpha):
    """Detecta automáticamente normal / t / skew-t"""
    if "norm" in base:
        return var_normal(mu, sigma, alpha), es_normal(mu, sigma, alpha)

    if "std" in base:
        return var_student(mu, sigma, shape, alpha), es_student(mu, sigma, shape, alpha)

    raise ValueError(f"No reconozco distribución en '{base}'")

def kupiec_lr(hits, alpha):
    hits = np.asarray(hits).astype(int)
    T = len(hits)
    x = hits.sum()

    pi_hat = x / T if T > 0 else 0
    eps = 1e-12

    term1 = x * np.log((pi_hat + eps)/(alpha + eps))
    term2 = (T - x) * np.log(((1 - pi_hat) + eps)/((1 - alpha) + eps))

    LR_uc = 2 * (term1 + term2)
    p_uc = chi2.sf(LR_uc, df=1)
    return LR_uc, p_uc


def christoffersen_independence(hits):
    hits = np.asarray(hits).astype(int)
    T = len(hits)

    n00 = np.sum((hits[:-1] == 0) & (hits[1:] == 0))
    n01 = np.sum((hits[:-1] == 0) & (hits[1:] == 1))
    n10 = np.sum((hits[:-1] == 1) & (hits[1:] == 0))
    n11 = np.sum((hits[:-1] == 1) & (hits[1:] == 1))

    eps = 1e-12
    denom0 = n00 + n01
    denom1 = n10 + n11

    if denom0 == 0 or denom1 == 0:
        return 0.0, 1.0

    p01 = n01 / denom0
    p11 = n11 / denom1
    p = (n01 + n11) / (T - 1)

    L0 = (n00 + n10) * np.log(1 - p + eps) + (n01 + n11) * np.log(p + eps)
    L1 = (
        n00 * np.log(1 - p01 + eps) +
        n01 * np.log(p01 + eps) +
        n10 * np.log(1 - p11 + eps) +
        n11 * np.log(p11 + eps)
    )

    LR_ind = -2 * (L0 - L1)
    p_ind = chi2.sf(LR_ind, df=1)
    return LR_ind, p_ind


def christoffersen_cc(hits, alpha):
    LR_uc, p_uc = kupiec_lr(hits, alpha)
    LR_ind, p_ind = christoffersen_independence(hits)
    LR_cc = LR_uc + LR_ind
    p_cc = chi2.sf(LR_cc, df=2)
    return LR_cc, p_cc


def garch_var_es_backtest(df, N=1194, alpha_levels=[0.05, 0.01]):
    VaR_results = {}
    ES_results = {}

    for col in df.columns:
        if col.endswith("_mu"):
            base = col[:-3]
            mu = df[col].iloc[-N:]
            sigma = df[f"{base}_sigma"].iloc[-N:]
            shape = df.get(f"{base}_shape")
            if shape is not None:
                shape = shape.iloc[-N:]

            for alpha in alpha_levels:
                VaR_series, ES_series = compute_var_es(mu, sigma, shape, base, alpha)

                name = f"{base}_VaR_{int(alpha*100)}"
                VaR_results[name] = VaR_series

                name_es = f"{base}_ES_{int(alpha*100)}"
                ES_results[name_es] = ES_series

    VaR_df = pd.DataFrame(VaR_results)
    ES_df = pd.DataFrame(ES_results)

    real = df["Realized_Ret"].iloc[-N:].values

    summary = []

    for col in VaR_df.columns:
        VaR_series = VaR_df[col].values
        hits = (real < -VaR_series).astype(int)
        alpha = int(col.split("_")[-1]) / 100.0

        exc = hits.sum()
        hr = hits.mean()

        LR_uc, p_uc = kupiec_lr(hits, alpha)
        LR_ind, p_ind = christoffersen_independence(hits)
        LR_cc, p_cc = christoffersen_cc(hits, alpha)

        summary.append({
            "model": col,
            "exceedances": exc,
            "hit_ratio": hr,

            "LR_uc": LR_uc,
            "p_uc": p_uc,

            "LR_ind": LR_ind,
            "p_ind": p_ind,

            "LR_cc": LR_cc,
            "p_cc": p_cc
        })

    Backtest_df = pd.DataFrame(summary).set_index("model")

    return VaR_df, ES_df, Backtest_df


In [ ]:
VaR_df, ES_df, Backtest_df = garch_var_es_backtest(garch_forecasts)

Backtest_df


In [ ]:
real = garch_forecasts["Realized_Ret"].iloc[-1194:]
exceedances = {}

for col in VaR_df.columns:
    hits = real < (-VaR_df[col])
    exceedances[col] = {
        "exceedances": hits.sum(),
        "hit_ratio": hits.mean()
    }

exceedances_df = pd.DataFrame(exceedances).T
exceedances_df


In [ ]:
def run_gru_wfv(
    df,
    feature_list,
    target_col="SvOLHC_n10",
    train_size=1008,
    window_size=504,
    n_steps=6,
    epochs=150,
    lr=0.0009
):

    y_df = df[[target_col]].shift(-1)

    df_model = pd.concat([df[feature_list], y_df], axis=1).dropna()

    X = df_model[feature_list].values
    y = df_model[target_col].values

    print(f"\nDataset limpio: {X.shape}, Features: {feature_list}")

    dataset = np.hstack((X, y.reshape(-1, 1)))
    n_features = X.shape[1]

    y_pred_total = []
    y_true_total = []

    for i in range(train_size, len(dataset), window_size):

        train_data = dataset[i-train_size:i]
        test_data  = dataset[i-n_steps:i+window_size]

        train_used = train_data
        test_used  = test_data

        X_train, y_train = split_sequences(train_used, n_steps)
        X_train = X_train.reshape((X_train.shape[0], n_steps, n_features))

        split = int(len(X_train) * 0.67)

        X_tr, X_val = X_train[:split], X_train[split:]
        y_tr, y_val = y_train[:split], y_train[split:]

        model = Sequential([
            GRU(512, activation="relu", return_sequences=True,
                kernel_regularizer=regularizers.l2(1e-5),
                input_shape=(n_steps, n_features)),
            Dropout(0.3),

            GRU(256, activation="relu", return_sequences=True,
                kernel_regularizer=regularizers.l2(1e-5)),
            Dropout(0.3),

            GRU(128, activation="relu",
                kernel_regularizer=regularizers.l2(1e-5)),
            Dropout(0.3),

            Dense(1)
        ])

        opt = Adam(learning_rate=lr)
        model.compile(optimizer=opt, loss="mse", metrics=["mae"])

        model_path = f"best_model_{i}.h5"

        mc = ModelCheckpoint(
            model_path,
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            verbose=0
        )

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            batch_size=500,
            epochs=epochs,
            verbose=0,
            callbacks=[mc]
        )

        best_model = load_model(model_path, compile=False)
        best_model.compile(optimizer=opt, loss="mse")

        X_test, y_test = split_sequences(test_used, n_steps)
        X_test = X_test.reshape((X_test.shape[0], n_steps, n_features))

        yhat = best_model.predict(X_test, verbose=0)

        y_pred_total.extend(yhat.flatten())
        y_true_total.extend(y_test.flatten())

    y_pred = np.array(y_pred_total)
    y_true = np.array(y_true_total)

    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)

    print("\nRESULTADOS FINALES")
    print(f"MSE: {mse:.8f} | MAE: {mae:.8f}")

    return y_true, y_pred, {"MSE": mse, "MAE": mae}

In [ ]:
garch_fe = garch_forecasts.copy()

# SHOCK FEATURES
garch_fe["Abs_Ret_sq"]  = garch_fe["Abs_Ret"] ** 2

# LAGS (1,2,3)
base_cols = ["SvOLHC", "Abs_Ret"]

for col in base_cols:
    for lag in [1, 2, 3]:
        garch_fe[f"{col}_lag{lag}"] = garch_fe[col].shift(lag)

# HAR
garch_fe["SvOLHC_roll_5_shift"] = (
    garch_fe["SvOLHC"].rolling(5).mean().shift(1)
)

garch_fe["SvOLHC_roll_10_shift"] = (
    garch_fe["SvOLHC"].rolling(10).mean().shift(1)
)

garch_fe["Abs_Ret_roll_5_shift"] = (
    garch_fe["Abs_Ret"].rolling(5).mean().shift(1)
)

garch_fe["Abs_Ret_roll_10_shift"] = (
    garch_fe["Abs_Ret"].rolling(10).mean().shift(1)
)

# DIFERENCIAS
garch_fe["SvOLHC_diff1_shift"] = (
    garch_fe["SvOLHC"].diff(1).shift(1)
)

garch_fe["Abs_Ret_diff1_shift"] = (
    garch_fe["Abs_Ret"].diff(1).shift(1)
)

# LOG
garch_fe["log_SvOLHC_shift"] = np.log(
    garch_fe["SvOLHC"].shift(1) + 1e-8
)

garch_fe["log_Abs_Ret_shift"] = np.log(
    garch_fe["Abs_Ret"].shift(1) + 1e-8
)

# REGIME (rolling 20)
SvOLHC_rolling_mean_20 = garch_fe["SvOLHC"].rolling(20).mean().shift(1)

garch_fe["SvOLHC_high_vol_regime_shift"] = (
    garch_fe["SvOLHC"].shift(1) > SvOLHC_rolling_mean_20
).astype(int)

Abs_Ret_rolling_mean_20 = garch_fe["Abs_Ret"].rolling(20).mean().shift(1)

garch_fe["Abs_Ret_high_vol_regime_shift"] = (
    garch_fe["Abs_Ret"].shift(1) > Abs_Ret_rolling_mean_20
).astype(int)

garch_fe = garch_fe.dropna().reset_index(drop=True)

print("Shape:", garch_fe.shape)

new_cols = [
    col for col in garch_fe.columns
    if col not in ["SvOLHC", "Abs_Ret", "sGARCH_std_sigma"]
]

print("\nNuevas columnas:")
print(new_cols)

In [ ]:
feature_sets = {
    "only_SvOLHC": [
        "SvOLHC"
    ],

    "only_returns": [
        "Abs_Ret"
    ],

    "only_garch": [
        "sGARCH_norm_sigma"
    ],

    "SvOLHC_returns": [
        "SvOLHC",
        "Abs_Ret"
    ],

    "SvOLHC_garch": [
        "SvOLHC",
        "sGARCH_norm_sigma"
    ],

    "returns_garch": [
        "Abs_Ret",
        "sGARCH_norm_sigma"
    ],

    "baseline_all": [
        "SvOLHC",
        "Abs_Ret",
        "sGARCH_norm_sigma"
    ]
}

feature_sets_test = {

    "baseline": [
        "SvOLHC",
        "Abs_Ret"
    ],

    "har_only": [
        "SvOLHC",
        "Abs_Ret",
        "SvOLHC_roll_5_shift",
        "SvOLHC_roll_10_shift",
        "Abs_Ret_roll_5_shift",
        "Abs_Ret_roll_10_shift"
    ],

    "lags_only": [
        "SvOLHC",
        "Abs_Ret",
        "SvOLHC_lag1", "SvOLHC_lag2", "SvOLHC_lag3",
        "Abs_Ret_lag1", "Abs_Ret_lag2", "Abs_Ret_lag3"
    ],

    "diff_only": [
        "SvOLHC",
        "Abs_Ret",
        "SvOLHC_diff1_shift",
        "Abs_Ret_diff1_shift"

    ],

    "shocks_only": [
        "SvOLHC",
        "Abs_Ret",
        "Abs_Ret_sq"
    ],

    "regime_only": [
        "SvOLHC",
        "Abs_Ret",
        "SvOLHC_high_vol_regime_shift",
        "Abs_Ret_high_vol_regime_shift"
    ],

    "log_only": [
        "SvOLHC",
        "Abs_Ret",
        "log_SvOLHC_shift",
        "log_Abs_Ret_shift"

    ]
}

feature_sets_best = {

    "baseline": [
        "SvOLHC",
        "Abs_Ret",
    ],

    "har_only": [
        "SvOLHC",
        "Abs_Ret",
        "SvOLHC_roll_5_shift",
        "SvOLHC_roll_10_shift",
        "Abs_Ret_roll_5_shift",
        "Abs_Ret_roll_10_shift"

    ],

    "diff_only": [
        "SvOLHC",
        "Abs_Ret",
        "SvOLHC_diff1_shift",
        "Abs_Ret_diff1_shift" 
    ]
}

In [ ]:
results = {}

for name, features in feature_sets_test.items():
    print(f"\nRunning: {name}")
    
    try:
        y_true, y_pred, metrics = run_gru_wfv(
            df=garch_fe,
            feature_list=features,
            train_size=1008,
            window_size=504,
            n_steps=6,
            epochs=150
        )
        
        results[name] = metrics
        
    except Exception as e:
        print(f"Error en {name}: {e}")
        results[name] = None

results_df = pd.DataFrame(results).T
print("\nResultados:")
print(results_df.sort_values(by=list(results_df.columns)[0]))